# Prepare AIME for prompt optimization

This notebook loads the complete historical source through 2024 and the normalized AIME 2025 source, shows five examples, applies a chronological split, validates the records, and writes JSONL plus combined JSON files under `data/processed/aime/`.

Split policy: train = 1983–2018, validation = 2019–2021, and test = 2022–2025. The test set therefore contains 120 problems.

## Dependency

Reading the historical Parquet file requires `pandas` and `pyarrow`. If needed, run this once in a notebook cell: `%pip install pandas pyarrow`.

In [1]:
import json
from pathlib import Path

import pandas as pd

In [2]:
def find_repo_root():
    """Find the repository root from the current notebook directory."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "AGENTS.md").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


def read_jsonl(path):
    """Read non-empty JSON objects from a JSONL file."""
    with path.open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def show_examples(records, count=5):
    """Display a small number of records in readable JSON."""
    print(json.dumps(records[:count], ensure_ascii=False, indent=2))

In [3]:
REPO_ROOT = find_repo_root()
HISTORICAL_PATH = REPO_ROOT / "data/aime_historical/sources/pandores_aime_1983_2025/data.parquet"
AIME_2025_PATH = REPO_ROOT / "data/aime_2025/aime2025.jsonl"
OUTPUT_DIR = REPO_ROOT / "data/processed/aime"

historical = pd.read_parquet(HISTORICAL_PATH)
historical = historical[historical["year"].astype(int) <= 2024].copy()
aime_2025 = read_jsonl(AIME_2025_PATH)

print(f"Historical 1983–2024: {len(historical):,} records")
print(f"AIME 2025: {len(aime_2025):,} records")

Historical 1983–2024: 1,005 records
AIME 2025: 30 records


## Five original examples

In [4]:
raw_example_fields = ["year", "part", "index", "problem", "answer"]
show_examples(historical[raw_example_fields].head(5).to_dict(orient="records"), count=5)

[
  {
    "year": 1983,
    "part": "AIME",
    "index": 1,
    "problem": "Let $x$, $y$ and $z$ all exceed $1$ and let $w$ be a positive number such that $\\log_x w = 24$, $\\log_y w = 40$ and $\\log_{xyz} w = 12$. Find $\\log_z w$.",
    "answer": 60
  },
  {
    "year": 1983,
    "part": "AIME",
    "index": 2,
    "problem": "Let $f(x)=|x-p|+|x-15|+|x-p-15|$, where $0 < p < 15$. Determine the minimum value taken by $f(x)$ for $x$ in the interval $p \\leq x\\leq15$.",
    "answer": 15
  },
  {
    "year": 1983,
    "part": "AIME",
    "index": 3,
    "problem": "What is the product of the real roots of the equation $x^2 + 18x + 30 = 2 \\sqrt{x^2 + 18x + 45}$?",
    "answer": 20
  },
  {
    "year": 1983,
    "part": "AIME",
    "index": 4,
    "problem": "A machine-shop cutting tool has the shape of a notched circle, as shown. The radius of the circle is $\\sqrt{50}$ cm, the length of $AB$ is $6$ cm and that of $BC$ is $2$ cm. The angle $ABC$ is a right angle. Find the square of the

In [5]:
def normalize_exam(value):
    """Convert historical exam labels to AIME, AIME I, or AIME II."""
    text = "" if pd.isna(value) else str(value).strip().upper()
    if text in {"I", "1", "AIME I"}:
        return "AIME I"
    if text in {"II", "2", "AIME II"}:
        return "AIME II"
    return "AIME"


def split_for_year(year):
    """Assign a chronological train, validation, or test split."""
    if year <= 2018:
        return "train"
    if year <= 2021:
        return "validation"
    return "test"


def make_aime_id(year, exam, problem_number):
    """Create a stable ID from the year, exam, and problem number."""
    exam_slug = exam.lower().replace(" ", "-")
    return f"aime-{year}-{exam_slug}-{problem_number:02d}"


def build_record(year, exam, problem_number, question, answer):
    """Create one normalized AIME prompt-optimization record."""
    year = int(year)
    problem_number = int(problem_number)
    answer = str(int(answer))
    return {
        "id": make_aime_id(year, exam, problem_number),
        "dataset": "aime",
        "task_type": "math_short_answer",
        "split": split_for_year(year),
        "year": year,
        "exam": exam,
        "problem_number": problem_number,
        "question": str(question).strip(),
        "answer": answer,
    }

In [6]:
records = []
for row in historical.to_dict(orient="records"):
    records.append(
        build_record(
            year=row["year"],
            exam=normalize_exam(row["part"]),
            problem_number=row["index"],
            question=row["problem"],
            answer=row["answer"],
        )
    )

for row in aime_2025:
    records.append(
        build_record(
            year=row["year"],
            exam=normalize_exam(row["exam"]),
            problem_number=row["problem_number"],
            question=row["question"],
            answer=row["answer"],
        )
    )

exam_order = {"AIME": 0, "AIME I": 1, "AIME II": 2}
records.sort(key=lambda row: (row["year"], exam_order[row["exam"]], row["problem_number"]))

In [7]:
def validate_aime(records):
    """Check uniqueness, answers, years, and expected split counts."""
    assert len(records) == 1035
    assert len({record["id"] for record in records}) == len(records)
    assert all(record["question"] and record["answer"].isdigit() for record in records)
    assert all(0 <= int(record["answer"]) <= 999 for record in records)
    counts = {split: sum(record["split"] == split for record in records) for split in ("train", "validation", "test")}
    assert counts == {"train": 825, "validation": 90, "test": 120}
    assert {record["year"] for record in records if record["split"] == "test"} == {2022, 2023, 2024, 2025}
    return counts


split_counts = validate_aime(records)
print("Validation passed:", split_counts)

Validation passed: {'train': 825, 'validation': 90, 'test': 120}


## Five prepared examples

In [8]:
show_examples(records, count=5)

[
  {
    "id": "aime-1983-aime-01",
    "dataset": "aime",
    "task_type": "math_short_answer",
    "split": "train",
    "year": 1983,
    "exam": "AIME",
    "problem_number": 1,
    "question": "Let $x$, $y$ and $z$ all exceed $1$ and let $w$ be a positive number such that $\\log_x w = 24$, $\\log_y w = 40$ and $\\log_{xyz} w = 12$. Find $\\log_z w$.",
    "answer": "60"
  },
  {
    "id": "aime-1983-aime-02",
    "dataset": "aime",
    "task_type": "math_short_answer",
    "split": "train",
    "year": 1983,
    "exam": "AIME",
    "problem_number": 2,
    "question": "Let $f(x)=|x-p|+|x-15|+|x-p-15|$, where $0 < p < 15$. Determine the minimum value taken by $f(x)$ for $x$ in the interval $p \\leq x\\leq15$.",
    "answer": "15"
  },
  {
    "id": "aime-1983-aime-03",
    "dataset": "aime",
    "task_type": "math_short_answer",
    "split": "train",
    "year": 1983,
    "exam": "AIME",
    "problem_number": 3,
    "question": "What is the product of the real roots of the equatio

In [9]:
def write_jsonl(path, records):
    """Write records as one JSON object per line."""
    with path.open("w", encoding="utf-8") as stream:
        for record in records:
            stream.write(json.dumps(record, ensure_ascii=False) + "\n")


def write_json(path, value):
    """Write a JSON value with readable indentation."""
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for split in ("train", "validation", "test"):
    split_records = [record for record in records if record["split"] == split]
    write_jsonl(OUTPUT_DIR / f"{split}.jsonl", split_records)

write_json(OUTPUT_DIR / "all.json", records)
write_json(
    OUTPUT_DIR / "dataset_info.json",
    {
        "dataset": "aime",
        "task_type": "math_short_answer",
        "split_years": {"train": "1983-2018", "validation": "2019-2021", "test": "2022-2025"},
        "splits": split_counts,
        "files": ["train.jsonl", "validation.jsonl", "test.jsonl", "all.json"],
    },
)
print(f"Saved {len(records):,} records to {OUTPUT_DIR}")

Saved 1,035 records to /storage2/home/aunabilchakma/codes/RE_Prompt_optimizer/data/processed/aime
